# Official workload tail behavior
This figure uses p95/p99 token summaries from the official SWE-Bench Lite and Terminal-Bench tasks. The old robustness files (`robustness.json`, `real_agent_robustness.json`) remain available for mechanism-only fault-injection plots and are not mixed into this application workload figure.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)

frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists():
        frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    view = (df.groupby(['suite', 'scale', 'mode'], as_index=False)[['total_tokens_p95', 'total_tokens_p99']].mean())
    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8), dpi=300, sharey=True)
    for ax, metric, title in zip(axes, ['total_tokens_p95', 'total_tokens_p99'], ['p95 tokens', 'p99 tokens']):
        for (suite, mode), group in view.groupby(['suite', 'mode']):
            group = group.sort_values('scale')
            ax.plot(group['scale'], group[metric], marker='o', linewidth=1.0, label=f'{suite}:{mode}')
        ax.set_title(title, fontsize=8)
        ax.set_xlabel('Task scale', fontsize=8)
        ax.tick_params(labelsize=7)
    axes[0].set_ylabel('Tokens', fontsize=8)
    axes[1].legend(fontsize=5.5, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Motivation-Official-Tail.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Motivation-Official-Tail.pdf', bbox_inches='tight')
else:
    print('No official summary found; run experiments/scripts/bench_official_tasks.py first.')
print('official workload rows:', len(df))
# Real-agent refactor is represented by the external harness boundary, not a synthetic trace.
# Legacy names retained for provenance: robustness.json, real_agent_robustness.json.
